# 04_01 Nearest neighbours: are you who your neighbours are?

K-nearest neighbours (KNN) classifies a sentence by finding the training sentences closest to it and letting
them vote. Everything depends on what "closest" means. By the end of this notebook you will have measured
four distances by hand, watched a model score 100 percent on its training data and 76 on new data, and found
that the meaning of a sentence is mostly not its sentiment.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-04-which-classifier-and-why", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import clftools
from nlpcheck import ask, guess, reveal, check_04_01

df = clftools.load_sentences()
X_train, X_test, y_train, y_test = clftools.split(df)
print(len(df), "sentences:", df["label"].value_counts().to_dict(), "| train", len(X_train), "test", len(X_test))
df.sample(5, random_state=1)

Each row is one sentence from a review on Amazon, IMDB or Yelp, labelled 1 for positive and 0 for negative by
the researchers who collected them. Half are positive. The split is the same in every notebook of this lab.

## 1. Recall

From Lab 03.

**r1.** What does recall measure? (a) of the real positives, how many the model caught, (b) of the model's
positives, how many were real, (c) the share of all answers that were right

**r2.** Why fit a vectorizer inside a `Pipeline`? (a) it is faster, (b) so the same transformation, fitted on
training data only, is used when the model predicts, (c) it removes stop words

In [ ]:
ask("r1", "")
ask("r2", "")

## 2. Four distances, by hand

KNN needs a way to measure how far apart two points are. The book gives three; text adds a fourth.

- **Euclidean**: the straight-line distance, the square root of the sum of squared differences.
- **Manhattan**: the sum of the absolute differences, like walking city blocks.
- **Hamming**: the number of positions that differ, for categories that have no size.
- **Cosine**: one minus the cosine of the angle between two vectors. It ignores length, which matters for text,
  where a long review and a short one can say the same thing.

In [ ]:
A, B = np.array([1, 3]), np.array([2, 3])
print("Euclidean A-B:", np.sqrt(((A - B) ** 2).sum()))
print("Manhattan A-B:", np.abs(A - B).sum())

p = ["red", "small", "round"]
q = ["red", "large", "round"]
print("Hamming p-q:  ", sum(a != b for a, b in zip(p, q)))

u, v = np.array([1.0, 1.0, 0.0]), np.array([3.0, 3.0, 0.0])
print("Cosine u-v:   ", 1 - u @ v / (np.linalg.norm(u) * np.linalg.norm(v)), "(same direction, three times longer)")

The first line is the book's assessment question: points (1, 3) and (2, 3) are 1 apart. The last line is the
reason text uses cosine: `v` is `u` said three times, and their cosine distance is 0.

## 3. Which sentence is nearer?

Turn three short phrases into word counts. Predict: by Euclidean distance on the counts, is "great phone"
nearer to "terrible phone" or to "excellent handset"?

In [ ]:
guess("nearer_to_great_phone", None)   # "terrible phone" or "excellent handset" 

In [ ]:
phrases = ["great phone", "terrible phone", "excellent handset"]
cv = CountVectorizer().fit(phrases)
V = cv.transform(phrases).toarray()
print(cv.get_feature_names_out())
print(V)
d_terrible = np.linalg.norm(V[0] - V[1])
d_excellent = np.linalg.norm(V[0] - V[2])
print(f"great phone to terrible phone:    {d_terrible:.2f}")
print(f"great phone to excellent handset: {d_excellent:.2f}")
reveal("nearer_to_great_phone", "terrible phone" if d_terrible < d_excellent else "excellent handset")

"Terrible phone" is nearer, 1.41 against 2.00, because counts only see shared words, and "phone" is shared.
"Great" and "excellent" are different columns, as unrelated as "great" and "invoice".

Sentence embeddings (Lab 02) were built to fix exactly this: they place sentences by meaning. Predict again:
by cosine similarity of embeddings, which is nearer now?

In [ ]:
guess("nearer_by_meaning", None)   # "terrible phone" or "excellent handset" 

In [ ]:
E = clftools.embed(phrases)
s_terrible, s_excellent = float(E[0] @ E[1]), float(E[0] @ E[2])
print(f"similarity great phone / terrible phone:    {s_terrible:.3f}")
print(f"similarity great phone / excellent handset: {s_excellent:.3f}")
reveal("nearer_by_meaning", "terrible phone" if s_terrible > s_excellent else "excellent handset")

Still "terrible phone", 0.864 against 0.839. Nothing is broken. An embedding places a sentence by everything
it means, and most of what "great phone" means is that it is about a phone; whether the writer liked it is
one small part. Two phone reviews are close whatever their verdict. Keep this in mind for section 5: KNN weighs
every direction of meaning equally, and sentiment is only one of them.

## 4. KNN on the sentences, and the K = 1 trap

Predict: with K = 1, what accuracy will KNN score **on its own training sentences**? (a number between 0 and 1)

In [ ]:
guess("k1_train_accuracy", None)

In [ ]:
rows = []
for k in [1, 3, 5, 9, 15, 25, 51, 101]:
    m = Pipeline([("tfidf", TfidfVectorizer()), ("knn", KNeighborsClassifier(n_neighbors=k))]).fit(X_train, y_train)
    rows.append((k, m.score(X_train, y_train), m.score(X_test, y_test)))
    print(f"K = {k:3}   train {rows[-1][1]:.3f}   test {rows[-1][2]:.3f}")
reveal("k1_train_accuracy", round(rows[0][1], 3))

ks, tr, te = zip(*rows)
plt.figure(figsize=(7, 3.5))
plt.plot(ks, tr, "o-", label="train"); plt.plot(ks, te, "o-", label="test")
plt.xscale("log"); plt.xlabel("K (log scale)"); plt.ylabel("accuracy"); plt.legend(); plt.title("KNN on TF-IDF")
plt.show()

With K = 1 every training sentence is its own nearest neighbour, at distance zero, so it votes for its own
label: 100 percent, and meaningless. On the test set K = 1 scores 0.759. As K grows, training accuracy falls
(the model can no longer memorise) and test accuracy first rises, then falls again as the vote averages over
so many neighbours that it drifts towards the majority. That is the bias and variance trade-off the chapter
describes, in one chart.

## 5. The same neighbours, measured by meaning

The sentence embeddings below come from `clftools.embed()`. Your session started computing them in the
background the moment it began, while you read the chapter, and stores each one under `out/.embeddings/`.
That takes about four minutes. If you got here sooner, this cell finishes the job itself, which can take up to
about six minutes on a session's computer if none of it was done; the line it prints says how many vectors were
already waiting. Once they are all cached, the whole notebook runs in well under half a minute.

In [ ]:
t = time.time()
print(clftools.cached(df["text"]), "of", len(df), "sentence vectors were already cached")
E_train, E_test = clftools.embed(X_train), clftools.embed(X_test)
print(f"embeddings ready in {time.time() - t:.1f} s")
for k in [1, 5, 15, 25, 51]:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine").fit(E_train, y_train)
    print(f"K = {k:3}   test {knn.score(E_test, y_test):.3f}")

Better, but not by much: the best is about 0.80 at K = 51, against 0.78 for TF-IDF. Section 3 explains why.
Neighbours by meaning are mostly neighbours by topic, and KNN cannot be told to pay more attention to the
sentiment direction. A model that is **trained**, like the logistic regression in the third notebook, can.

This is still the most important idea in the notebook for the rest of the course: finding the nearest
vectors to a query is exactly what a vector database does, at the scale of millions, with an index that
finds approximately nearest neighbours without comparing against every one.

## 6. Your turn: choose K by cross-validation

Section 4 chose K by looking at the **test** set, which is cheating: the test set is supposed to be unseen. The
honest way is to split the **training** set into five folds, score each K on each fold in turn, and pick the K
with the best average. `GridSearchCV` does exactly that.

Build a `Pipeline` of `TfidfVectorizer()` and `KNeighborsClassifier(metric="cosine")`, search
`n_neighbors` over `[1, 5, 15, 25, 51, 101]` with `cv=5` on the training set, then fill in the three values.

In [ ]:
grid = [1, 5, 15, 25, 51, 101]
pipe = Pipeline([("tfidf", TfidfVectorizer()), ("knn", KNeighborsClassifier(metric="cosine"))])
search = None          # YOUR CODE HERE: GridSearchCV(pipe, {"knn__n_neighbors": grid}, cv=5), fitted on the training set

best_k = None          # YOUR CODE HERE: the K the search chose
cv_accuracy = None     # YOUR CODE HERE: its mean accuracy over the five folds (best_score_)
test_accuracy = None   # YOUR CODE HERE: the chosen model's accuracy on X_test, y_test

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump({"best_k": best_k, "cv_accuracy": cv_accuracy, "test_accuracy": test_accuracy},
          open("out/04_01_knn.json", "w"), indent=1, default=float)
check_04_01()

## 7. Exit ticket

From the book's assessment.

**x1.** Which distance suits categorical features in KNN? (a) Euclidean, (b) Manhattan, (c) Hamming

**x2.** When K increases, bias: (a) increases, (b) decreases, (c) cannot say

In [ ]:
ask("x1", "")
ask("x2", "")

Explain it back: why did KNN on embeddings barely beat KNN on word counts for sentiment, when embeddings
"understand meaning"? One or two sentences.

*Your explanation:* 